### Processar o arquivo de base com as perguntas.

In [17]:
# Carregar as questões a serem usadas, cuja lógica está em outro arquivo.
# Realizar download do arquivo direto do GitHub.
!wget https://raw.githubusercontent.com/eduoududu/juridico/refs/heads/main/questions/open_questions.ipynb -O questions.ipynb
# Executar o notebook de base na íntegra.
%run questions.ipynb

--2026-05-04 23:03:56--  https://raw.githubusercontent.com/eduoududu/juridico/refs/heads/main/questions/open_questions.ipynb
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 32015 (31K) [text/plain]
Saving to: ‘questions.ipynb’

questions.ipynb     100%[===================>]  31.26K  --.-KB/s    in 0s      

2026-05-04 23:03:56 (99.0 MB/s) - ‘questions.ipynb’ saved [32015/32015]

Sucesso: open_questions.jsonl salvo.


### Instalação de pacotes, seguido de importação de bibliotecas e funções necessárias.

In [19]:
# Instalar ou atualizar as bibliotecas necessárias.
!pip install -q -U openai bert-score

import pandas as pd
import os
import re
import time
import logging
from typing import List, Dict
from google.colab import files
from openai import OpenAI
from google import genai
from google.colab import userdata
from bert_score import score

# Configuração de Log
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

### Utiltários para processamento de texto.



In [21]:
class TextUtility:
    """Ferramentas utilitárias para processamento de texto."""
    @staticmethod
    def clean_prefix(text: str) -> str:
        return re.sub(r'^\d+_?', '', text).replace('_', ' ')

    @staticmethod
    def count_occurrences(text: str, keywords: List[str]) -> int:
        if not text: return 0
        pattern = r'\b(' + '|'.join(keywords) + r')\b'
        return len(re.findall(pattern, str(text).lower()))

### Classes para IA.

In [22]:
class AIClientFactory:
    """Fábrica para instanciar clientes de diferentes provedores de IA."""
    @staticmethod
    def get_client(provider_key: str):
        api_key = userdata.get(provider_key)

        if provider_key == 'groq_api_key':
            return OpenAI(base_url="https://api.groq.com/openai/v1", api_key=api_key)
        elif provider_key == 'google_api_key':
            return genai.Client(api_key=api_key)
        elif provider_key == 'openrouter_api_key':
            return OpenAI(base_url="https://openrouter.ai/api/v1/", api_key=api_key)
        else:
            raise ValueError(f"Provedor {provider_key} não suportado.")

class PromptHandler:
    """Gere a criação de prompts estruturados para questões jurídicas."""
    @staticmethod
    def to_markdown(data: Dict) -> str:
        markdown_sections = []
        try:
            for key, value in data.items():
                markdown_sections.append(f"### {str(key)}:\n{str(value)}\n")
        except:
            TypeError("Erro ao montar o markdown.")
        return "\n".join(markdown_sections).strip()

class QuestionProcessor:
    """Classe principal para processar e submeter questões à IA."""
    def __init__(self, provider_key: str, model_id: str, max_tokens: int = 4096):
        self.client = AIClientFactory.get_client(provider_key)
        self.model_id = model_id
        self.provider_key = provider_key
        self.max_tokens = max_tokens
        self.results = []

    def Guardar(self, prompt: str) -> Dict:
        start_time = time.time()

        # Lógica para clientes baseados em OpenAI (Groq/OpenRouter)
        if hasattr(self.client, 'chat'):
            completion = self.client.chat.completions.create(
                model=self.model_id,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.1
            )
            response_text = completion.choices[0].message.content
        else:
            # Placeholder para lógica específica do Google GenAI
            response_text = "Resposta do modelo Google"

        duration = (time.time() - start_time) * 1000
        return {"content": response_text, "latency_ms": duration}

    def _submit_single(self, prompt: str) -> Dict:
        start_time = time.time()

        # Lógica para Google Gemini (Google GenAI SDK)
        if self.provider_key == 'google_api_key':
            response = self.client.models.generate_content(
                model=self.model_id,
                contents=prompt,
                config={
                    "temperature": 0.1,
                    "max_output_tokens": self.max_tokens,
                    "top_p": 0.95
                }
            )
            # Acessando a estrutura específica do Gemini
            response_text = response.candidates[0].content.parts[0].text

        # Lógica para OpenAI / Groq / OpenRouter
        else:
            completion = self.client.chat.completions.create(
                model=self.model_id,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.1,
                max_tokens=self.max_tokens
            )
            response_text = completion.choices[0].message.content

        duration_ms = (time.time() - start_time) * 1000
        return {"content": response_text, "latency_ms": duration_ms}

    def run_inference(self, df_questions: pd.DataFrame):
        for _, row in df_questions.iterrows():
            prompt_data = {
                'Pessoa': row.get('system', ''),
                'Categoria': row.get('category', ''),
                'Dificuldade': row.get('level', ''),
                'Contexto': row.get('statement', ''),
                'Instrução': row.get('turns', '')
            }

            prompt = PromptHandler.to_markdown(prompt_data)
            result = self._submit_single(prompt)

            self.results.append({
                'question_id': row.get('num'),
                'response': result['content'],
                'time': result['latency_ms']
            })
            logging.info(f"Questão {row.get('question')} finalizada.")

    def get_results_df(self) -> pd.DataFrame:
        return pd.DataFrame(self.results)

class OutputManager:
    """Gere o salvamento e exportação de dados."""
    @staticmethod
    def save_to_jsonl(df: pd.DataFrame, filename: str):
        try:
            df.to_json(filename, orient='records', lines=True, force_ascii=False)
            logging.info(f"Arquivo {filename} salvo com sucesso.")
        except Exception as e:
            logging.error(f"Erro ao guardar JSONL: {e}")

### Realizar consulta usando o modelo llama 3 de 8 bilhões de parâmetros.

In [23]:
# Modelo llama 3.1 de 8 bilhões de parâmetros.
model_id1 = 'llama-3.1-8b-instant'
ia1 = QuestionProcessor('groq_api_key', model_id1)
ia1.run_inference(df_my_questions)
df_ia1 = ia1.get_results_df()
OutputManager.save_to_jsonl(df_ia1, 'resultados_ia1.jsonl')

### Realizar consulta usando o modelo llama 4 de 17 bilhões de parâmetros.

In [24]:
# Modelo llama 4 de 17 bilhões de parâmetros.
model_id2 = 'meta-llama/llama-4-scout-17b-16e-instruct'
ia2 = QuestionProcessor('groq_api_key', model_id2, max_tokens=8192)
ia2.run_inference(df_my_questions)
df_ia2 = ia2.get_results_df()
OutputManager.save_to_jsonl(df_ia2, 'resultados_ia2.jsonl')

### Realizar consulta usando o modelo gemini 3 flash lite preview, lançado em 2026.

In [ ]:
# O modelo escolhi do para rodar é o Gemini da Google em nuvém preview| de 2026.
model_id3 = 'gemini-3.1-flash-lite-preview'
ia3 = QuestionProcessor('google_api_key', model_id3)
ia3.run_inference(df_my_questions)
df_ia3 = ia3.get_results_df()
OutputManager.save_to_jsonl(df_ia3, 'resultados_ia3.jsonl')

### Classe para avaliação quantitativa das respostas usando a métrica BERR score.

In [ ]:
class MetricsEvaluator:
    """Responsável por avaliar a qualidade das respostas usando NLP e consolidar referências."""

    def __init__(self, lang="pt"):
        self.lang = lang

    def evaluate_dataframe(self, results_df: pd.DataFrame, original_df: pd.DataFrame) -> list:
        """
        Orquestra o cálculo comparando as respostas da IA com as referências consolidadas.
        """
        candidates = results_df['response'].tolist()
        references = self._consolidate_references(original_df)

        if len(candidates) != len(references):
            logging.error("Erro: A quantidade de respostas e referências não coincide.")
            return [0.0] * len(candidates)

        try:
            logging.info("Iniciando cálculo do BERTScore...")
            P, R, F1 = score(candidates, references, lang=self.lang, verbose=False)
            return [f * 100 for f in F1.tolist()]
        except Exception as e:
            logging.error(f"Erro ao calcular BERTScore: {e}")
            return [0.0] * len(candidates)

Calcular a métrica BERTStore para o Dataframe das respostas das IAs.

In [ ]:
df_ias['llama3_bert'] = pd.Series(calculate_bert(df_ias['llama3'].tolist(), df_ias['choices'].tolist())).round(2)
df_ias['llama4_bert'] = pd.Series(calculate_bert(df_ias['llama4'].tolist(), df_ias['choices'].tolist())).round(2)
df_ias['gemini_bert'] = pd.Series(calculate_bert(df_ias['gemini'].tolist(), df_ias['choices'].tolist())).round(2)

In [ ]:
# Níveis de dificuldade.
difficulty_map = {
    1: 'Estagiário',
    2: 'Analista',
    3: 'Juiz',
    4: 'Ministro'
}

df_ias['level_name'] = df_ias['level'].map(difficulty_map)

Avaliar as respostas por um juiz on-line (IA).

In [ ]:
def llm_judge(client_ai, model_id, name_ia):
  # Criar lista vazia para guardar as respostas.
  responses = []

  # Repetição para percorrer todo Dataframe.
  for index, row in df_ias.iterrows():
    # preenchimento dos parâmetros da pergunta, com base na linha corrente.
    questao = row['question']
    pergunta = row['statement'] + ' ' + row['turns']
    baseline = row['choices']
    ia = row[name_ia]

    prompt_usuario = f"""
    Você é um jurista sênior especializado em Direito Administrativo e Tributário brasileiro.
    Sua função é atuar como juiz em um benchmark de IAs.
    Analise as respostas comparando-as rigorosamente com a Resposta Base (Gabarito).
    Atribua um percentual de acerto (0-100%). Traga apensa um número com duas casas decimais.
    PERGUNTA: {pergunta}

    RESPOSTA BASE (GABARITO): {baseline}
    ---
    RESPOSTA IA: {ia}
      ---
    [FORMATO DE RESPOSTA]
    Apenas um número com duas casas decimais.
    """
    # Realizar consulta a IA.
    completion = client_ai.chat.completions.create(
        model=model_id,
        messages=[
            {"role": "user", "content": prompt_usuario}
        ],
        temperature=0.1 # Conservador
    )
    response = completion.choices[0].message.content

    # Armazenar o resultado em uma lista.
    responses.append({'question': questao, 'response': response})
    print(f"Questão {questao} processada com sucesso.")

  # Fechar conexão com a IA (somente se você a usou ativamente)
  client_ai.close()

  # por motivo de performance as repostas foram adicionadas a uma lista.
  # No retorno a lista é convertida para um dataframe.
  return pd.DataFrame(responses)

In [ ]:
#model_id = "nvidia/nemotron-3-super-120b-a12b:free"
#df_score_llama3 = llm_judge(client_ai_instance('oprouter_api_key'), model_id, 'llama3')
#df_score_llama4 = llm_judge(client_ai_instance('oprouter_api_key'), model_id, 'llama4')
#df_score_gemini = llm_judge(client_ai_instance('oprouter_api_key'), model_id, 'gemini')

In [ ]:
from IPython.display import HTML
from datetime import date
import matplotlib.pyplot as plt
import io
import base64

# Get today's date
today_date = date.today().strftime('%d/%m/%Y')

# Part 1: Generate the table HTML
table_html_df_display = df_ias[['question', 'level_name', 'llama3_bert', 'llama4_bert', 'gemini_bert']].copy()

# Rename columns for display
table_html_df_display.rename(columns={
    'question': 'Questão',
    'level_name': 'Dificuldade',
    'llama3_bert': 'Llama3 (%)',
    'llama4_bert': 'Llama4 (%)',
    'gemini_bert': 'Gemini (%)'
}, inplace=True)

table_html_string = table_html_df_display.to_html(index=False, classes='', escape=False)

# Part 2: Generate the chart and embed as base64
plt.figure(figsize=(12, 6))
plt.plot(df_ias['question'], df_ias['llama3_bert'], label='Llama3 BERT Score')
plt.plot(df_ias['question'], df_ias['llama4_bert'], label='Llama4 BERT Score')
plt.plot(df_ias['question'], df_ias['gemini_bert'], label='Gemini BERT Score')
plt.title('BERT Score Comparativo entre Modelos de IA')
plt.xlabel('Questão')
plt.ylabel('BERT Score (%)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.xticks(df_ias['question']) # Ensure all question numbers are shown
plt.tight_layout()

buf = io.BytesIO()
plt.savefig(buf, format='png', bbox_inches='tight')
plt.close() # Close the plot to prevent it from displaying directly in the notebook output
image_base64 = base64.b64encode(buf.getvalue()).decode('utf-8')
chart_html_bert = f'<img src="data:image/png;base64,{image_base64}" alt="BERT Score Chart" style="max-width:100%; height:auto;">'

# Part 3: Integrate into the main HTML content
html_report = f"""
<html>
    <head>
        <style>
            body {{ font-family: sans-serif; line-height: 1.6; color: #333; margin: 20px; background-color: #f8f8f8; }}
            .container {{ max-width: 800px; margin: auto; background: #fff; padding: 30px; border-radius: 8px; box-shadow: 0 0 15px rgba(0,0,0,0.1); }}
            h1 {{ color: #0056b3; border-bottom: 2px solid #0056b3; padding-bottom: 10px; margin-bottom: 20px; text-align: center; }}
            h2 {{ color: #0056b3; margin-top: 30px; border-bottom: 1px solid #eee; padding-bottom: 5px; }}
            ul {{ list-style-type: disc; margin-left: 20px; padding-left: 0; }}
            li {{ margin-bottom: 10px; }}
            strong {{ color: #0056b3; }}
            .accuracy-box {{ background-color: #e6f7ff; border: 1px solid #b3e0ff; padding: 15px; border-radius: 5px; text-align: center; margin-top: 20px; font-size: 1.2em; }}
            .accuracy-value {{ font-size: 1.8em; font-weight: bold; color: #007bff; }}
            .plot-container {{ text-align: center; margin-top: 20px; }}
            .plot-container img {{ max-width: 100%; height: auto; border: 1px solid #ddd; border-radius: 5px; padding: 5px; background: #fff; }}
            .info-block {{ background-color: #f0f8ff; border: 1px solid #cceeff; padding: 15px; border-radius: 5px; margin-top: 20px; margin-bottom: 20px; font-size: 0.95em; line-height: 1.5; }}
            .info-block p {{ margin: 5px 0; }}
            /* Table Styling */
            .dataframe {{ border-collapse: collapse; margin-top: 20px; width: 100%;}}
            .dataframe th, .dataframe td {{ border: 1px solid #ddd; padding: 8px; text-align: left; }}
            .dataframe thead th {{
                background-color: #00008B; /* Dark blue */
                color: white; /* White text for contrast */
            }}
            .dataframe tbody tr {{ background-color: transparent !important; }}
            .dataframe tr:hover {{ background-color: #f1f1f1; }}
        </style>
    </head>
    <body>
        <div class="container">
            <h1>Avaliação de respostas de questões subjetivas de exames passados da OAB por um modelo de IA</h1>

            <div class="info-block">
                <p><strong>Aluno:</strong> Eduardo Henrique</p>
                <p><strong>Data de realização de teste:</strong> {today_date}</p>
                <p><strong>Código fonte:</strong> <a href="https://github.com/eduoududu/juridico" target="_blank">https://github.com/eduoududu/juridico</a></p>
            </div>

            <h2>Etapas Realizadas:</h2>
            <ul>
                <li><strong>Importação do conjunto de dados de perguntas da OAB:</strong> o conjunto de dados de perguntas de provas passadas da OAB foi importado do GitHub, disponível publicamente no endereço web: <a href="https://github.com/maritaca-ai/oab-bench/blob/main/data/judge_prompts.jsonl" target="_blank">https://github.com/maritaca-ai/oab-bench/blob/main/data/judge_prompts.jsonl</a>.</li>
                <li><strong>Importação do dataset de linha guia:</strong> o dataset de linha guia, contendo as respostas relacionadas às perguntas, foi importado do mesmo projeto do GitHub, pela URL: <a href="https://github.com/maritaca-ai/oab-bench/blob/main/data/oab_bench/reference_answer/guidelines.jsonl" target="_blank">https://github.com/maritaca-ai/oab-bench/blob/main/data/oab_bench/reference_answer/guidelines.jsonl</a>.</li>
                <li><strong>Adição de números às questões:</strong> foram adicionados números às questões para facilitar a leitura humana, considerando que o contador em listas no Python inicia do zero.</li>
                <li><strong>Análise e classificação do nível de dificuldade:</strong> foi realizada uma análise manual e interativa, com o auxílio de IA, para classificar o nível de dificuldade das questões, e essa informação foi incluída no conjunto de perguntas.</li>
                <li><strong>Interseção entre dataframes:</strong> foi realizada uma interseção entre os dataframes de questões e de linha guia para reunir todas as informações relevantes em um único dataframe.</li>
                <li><strong>Seleção de subconjunto de questões:</strong> por fim, um subconjunto das questões, especificamente da questão 31 à 40, foi retirado para análises mais focadas.</li>
                <li><strong>Organização em Notebooks Separados:</strong> para otimização e organização, o arquivo responsável pela extração, limpeza e carga dos dados foi separado em um notebook à parte.</li>
                <li><strong>Processamento de Questões por IA:</strong> outro notebook utilizou os dados do primeiro e percorreu todo o conjunto, submetendo as questões a diferentes modelos de IA.</li>
                <li><strong>Estruturação da Pergunta em Markdown:</strong> a pergunta foi cuidadosamente estruturada em formato Markdown para maior detalhamento do conteúdo a ser enviado a uma IA via chave de API.</li>
                <li><strong>Conteúdo do Markdown:</strong> o prompt Markdown incluiu os seguintes elementos: papel, categoria, contexto, dificuldade e instrução.</li>
                <li><strong>Observações sobre Questões Tipo 'Peça':</strong> em análise manual dos dados, foi percebido que, do conjunto de 10 linhas, duas são peças (informação retirada do campo 'statement'). Nesses casos, o campo 'turns' (instrução) estava vazio, o que pode indicar que não se trata de uma questão tradicional. Mesmo assim, foram submetidas às IAs para observar o comportamento.</li>
                <li><strong>Modelos de IA Utilizados:</strong> as questões foram enviadas a três modelos de IA: <b>llama-3.1-8b-instant</b>, <b>meta-llama/llama-4-scout-17b-16e-instruct</b> e <b>gemini-3.1-flash-lite-preview</b>. Esses modelos foram escolhidos por suas características e pela possibilidade de uso gratuito nesta atividade.</li>
                <li><strong>Métrica de Avaliação: BERTScore:</strong> métrica quantitativa utilizada para avaliar a acurácia das respostas de cada IA, comparando-as com a linha guia.</li>
                <li><strong>Tentativa de Avaliação por Juiz Online (IA):</strong> foi feita uma tentativa de utilizar um 'Juiz Online', uma quarta IA robusta, para avaliar as respostas. No entanto, as restrições de uso de IAs gratuitas impossibilitaram essa abordagem.</li>
                <li><strong>Relatório Comparativo Final:</strong> ao final, foi gerado um relatório comparativo utilizando os resultados do BERTScore.</li>
            </ul>
            <h2>Resultados Encontrados:</h2>
            <div class="plot-container">
                <h3>Gráfico de Acurácia Geral</h3>
                {chart_html_bert}
            </div>
            <h2>Pontuação das Respostas namétrica BERTScore</h2>
            {table_html_string}
        </div>
    </body>
</html>
"""
HTML(html_report)

In [ ]:
with open('/content/relatorio_analise_ia.html', 'w') as f:
    f.write(html_report)
print('Report saved to /content/relatorio_analise_ia.html')

In [ ]:
df_my_questions

In [ ]:
df_my_questions.iloc[0]["statement"]

In [ ]:
import pandas as pd



In [ ]:
df_my_questions

In [18]:
# df_my_questions

,num,question_id,edition,year,statement,turns,category,system,choices,level,type
30,31,39_direito_tributario_peca_profissional,39,2023,"PEÇA PRÁTICO-PROFISSIONAL\n\nPedro de Camões, ...",,Direito Tributário,Você é um bacharel em direito que está realiza...,"[{'index': 0, 'turns': ['A medida judicial cab...",4,Aberta
31,32,39_direito_tributario_questao_1,39,2023,QUESTÃO\n\nA sociedade empresária Metalúrgica ...,"A partir de quando se deve contar, no caso con...",Direito Tributário,Você é um bacharel em direito que está realiza...,"[{'index': 0, 'turns': ['O prazo para oferta d...",3,Aberta
32,33,39_direito_tributario_questao_2,39,2023,QUESTÃO\n\nA sociedade empresária Tudo Verde L...,A qual dos municípios o ISS de jardinagem é de...,Direito Tributário,Você é um bacharel em direito que está realiza...,"[{'index': 0, 'turns': ['O ISS de jardinagem é...",3,Aberta
33,34,39_direito_tributario_questao_3,39,2023,QUESTÃO\n\nA Administração Tributária Federal ...,É válida a exigência de depósito do montante c...,Direito Tributário,Você é um bacharel em direito que está realiza...,"[{'index': 0, 'turns': ['Não, pois é inconstit...",2,Aberta
34,35,39_direito_tributario_questao_4,39,2023,QUESTÃO\n\nA sociedade empresária Faz Tudo Ltd...,Está correto o argumento da necessidade de not...,Direito Tributário,Você é um bacharel em direito que está realiza...,"[{'index': 0, 'turns': ['Não está correto, por...",1,Aberta
35,36,40_direito_administrativo_peca_profissional,40,2024,PEÇA PRÁTICO-PROFISSIONAL\n\nO Ministério Públ...,,Direito Administrativo,Você é um bacharel em direito que está realiza...,"[{'index': 0, 'turns': ['O(a) examinando(a) de...",4,Aberta
36,37,40_direito_administrativo_questao_1,40,2024,QUESTÃO\n\nA sociedade empresária Sagaz S.A. e...,Há necessidade de demonstração do elemento sub...,Direito Administrativo,Você é um bacharel em direito que está realiza...,"[{'index': 0, 'turns': ['Não. A responsabiliza...",4,Aberta
37,38,40_direito_administrativo_questao_2,40,2024,QUESTÃO\n\nDeterminada informação de interesse...,É lícita a cobrança efetuada pelo órgão respon...,Direito Administrativo,Você é um bacharel em direito que está realiza...,"[{'index': 0, 'turns': ['Não. A submissão e o ...",4,Aberta
38,39,40_direito_administrativo_questao_3,40,2024,QUESTÃO\n\nCerta Secretaria do Estado Alfa fez...,É possível a utilização do sistema de registro...,Direito Administrativo,Você é um bacharel em direito que está realiza...,"[{'index': 0, 'turns': ['Sim. A Administração ...",4,Aberta
39,40,40_direito_administrativo_questao_4,40,2024,"QUESTÃO\n\nRecentemente, Iná foi aprovada em c...",A aprovação de Iná no mencionado concurso impo...,Direito Administrativo,Você é um bacharel em direito que está realiza...,"[{'index': 0, 'turns': ['Não. Iná foi aprovada...",1,Aberta
